In [4]:
# INSTALL PYSPARK

!pip install pyspark

In [5]:
# UPLOAD FILES

from google.colab import files
uploaded = files.upload()

# Upload:
# customers.csv
# orders.csv

Saving week3_customers.csv to week3_customers (2).csv
Saving week3_orders.csv to week3_orders (2).csv


In [6]:
# CREATE SPARK SESSION

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
spark = SparkSession.builder.appName("CustomerOrderAnalysis").getOrCreate()


In [7]:
# LOAD CUSTOMERS DATA
customers_df = spark.read.csv("week3_customers.csv",header=True,inferSchema=True)


In [8]:
# LOAD ORDERS DATA
orders_df = spark.read.csv("week3_orders.csv",header=True,inferSchema=True)

In [9]:
# SHOW CUSTOMERS DATA
print("CUSTOMERS DATA")
customers_df.show()

CUSTOMERS DATA
+-----------+-------------+------+
|customer_id|customer_name|region|
+-----------+-------------+------+
|          1|        Selva| South|
|          2|         Arun| North|
|          3|        Priya|  West|
|          4|        Divya|  East|
|          5|        Kumar| South|
|          6|         Ravi| North|
|          7|          Anu|  West|
|          8|        Vijay|  East|
+-----------+-------------+------+



In [10]:
# SHOW ORDERS DATA
print("ORDERS DATA")
orders_df.show()

ORDERS DATA
+--------+-----------+------------+----------+-------------+-------------+-------+
|order_id|customer_id|product_name|order_date|delivery_date|delivery_days|delayed|
+--------+-----------+------------+----------+-------------+-------------+-------+
|     101|          1|      Laptop|2026-05-01|   2026-05-10|            9|      1|
|     102|          1|       Mouse|2026-05-03|   2026-05-05|            2|      0|
|     103|          2|    Keyboard|2026-05-02|   2026-05-12|           10|      1|
|     104|          3|      Mobile|2026-05-04|   2026-05-06|            2|      0|
|     105|          4|      Tablet|2026-05-05|   2026-05-15|           10|      1|
|     106|          5|     Monitor|2026-05-06|   2026-05-08|            2|      0|
|     107|          6|     Printer|2026-05-07|   2026-05-20|           13|      1|
|     108|          7|     Speaker|2026-05-08|   2026-05-09|            1|      0|
|     109|          8|      Camera|2026-05-09|   2026-05-18|            9| 

In [11]:
# JOIN TABLES

joined_df = orders_df.join(customers_df,on="customer_id",how="inner")
print("JOINED DATA")
joined_df.show()

JOINED DATA
+-----------+--------+------------+----------+-------------+-------------+-------+-------------+------+
|customer_id|order_id|product_name|order_date|delivery_date|delivery_days|delayed|customer_name|region|
+-----------+--------+------------+----------+-------------+-------------+-------+-------------+------+
|          1|     101|      Laptop|2026-05-01|   2026-05-10|            9|      1|        Selva| South|
|          1|     102|       Mouse|2026-05-03|   2026-05-05|            2|      0|        Selva| South|
|          2|     103|    Keyboard|2026-05-02|   2026-05-12|           10|      1|         Arun| North|
|          3|     104|      Mobile|2026-05-04|   2026-05-06|            2|      0|        Priya|  West|
|          4|     105|      Tablet|2026-05-05|   2026-05-15|           10|      1|        Divya|  East|
|          5|     106|     Monitor|2026-05-06|   2026-05-08|            2|      0|        Kumar| South|
|          6|     107|     Printer|2026-05-07|   202

In [12]:
# FILTER DELAYED ORDERS
delayed_df = joined_df.filter(col("delayed")== 1)
print("DELAYED ORDERS")
delayed_df.show()


DELAYED ORDERS
+-----------+--------+------------+----------+-------------+-------------+-------+-------------+------+
|customer_id|order_id|product_name|order_date|delivery_date|delivery_days|delayed|customer_name|region|
+-----------+--------+------------+----------+-------------+-------------+-------+-------------+------+
|          1|     101|      Laptop|2026-05-01|   2026-05-10|            9|      1|        Selva| South|
|          2|     103|    Keyboard|2026-05-02|   2026-05-12|           10|      1|         Arun| North|
|          4|     105|      Tablet|2026-05-05|   2026-05-15|           10|      1|        Divya|  East|
|          6|     107|     Printer|2026-05-07|   2026-05-20|           13|      1|         Ravi| North|
|          8|     109|      Camera|2026-05-09|   2026-05-18|            9|      1|        Vijay|  East|
+-----------+--------+------------+----------+-------------+-------------+-------+-------------+------+



In [13]:
# REGION-WISE DELAY COUNT
region_delay_count = delayed_df.groupBy("region").count()

print("REGION-WISE DELAY COUNT")
region_delay_count.show()

REGION-WISE DELAY COUNT
+------+-----+
|region|count|
+------+-----+
| South|    1|
|  East|    2|
| North|    2|
+------+-----+



In [14]:
# SAVE OUTPUT
region_delay_count.write.mode("overwrite").option("header",True).csv("delays_by_region")
print("OUTPUT SAVED SUCCESSFULLY")

OUTPUT SAVED SUCCESSFULLY


In [15]:
# STOP SPARK SESSION
spark.stop()